In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

###  A simple convnet that achieves ~99% test accuracy on MNIST.

In [2]:
df_Train = pd.read_csv('../input/fashionmnist/fashion-mnist_train.csv')

In [3]:
df_Test = pd.read_csv('../input/fashionmnist/fashion-mnist_test.csv')

In [4]:
df = pd.concat([df_Train,df_Test])

In [5]:
df_Test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 59.9 MB


In [6]:
df_Train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 359.3 MB


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 70000 entries, 0 to 9999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 419.8 MB


In [8]:
from tensorflow import keras
from tensorflow.keras import layers

2021-10-06 18:51:02.559072: W tensorflow/stream_executor/platform/default/dso_loader.cc:60] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/conda/lib
2021-10-06 18:51:02.559176: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


# prepare the data

In [9]:
#model / data parameters
num_classes = 10
input_shape = (28,28,1) #grayscale

#split the data
(x_train,y_train),(x_test,y_test) = keras.datasets.mnist.load_data()

11493376/11490434 [==============================] - 0s 0us/step


In [10]:
#scale images to the [0,1] range

x_train = x_train.astype('float32')/255
x_test = x_test.astype('float32')/255




In [11]:
# make sure images has the shape (28,28,1)

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


In [12]:
#convert class vectors to binary class matrices

y_train = keras.utils.to_categorical(y_train,num_classes)
y_test = keras.utils.to_categorical(y_test,num_classes)


# build the model

In [13]:
model = keras.Sequential(
[
    keras.Input(shape=input_shape),
    layers.Conv2D(32, kernel_size=(3,3),activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Conv2D(64,kernel_size=(3,3),activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),   
]
)

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 13, 13, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 11, 11, 64)        18496     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 5, 5, 64)          0         
_________________________________________________________________
flatten (Flatten)            (None, 1600)              0         
_________________________________________________________________
dropout (Dropout)            (None, 1600)              0         
_________________________________________________________________
dense (Dense)                (None, 10)                1

2021-10-06 18:51:07.852503: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2021-10-06 18:51:07.857311: W tensorflow/stream_executor/platform/default/dso_loader.cc:60] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/conda/lib
2021-10-06 18:51:07.857341: W tensorflow/stream_executor/cuda/cuda_driver.cc:326] failed call to cuInit: UNKNOWN ERROR (303)
2021-10-06 18:51:07.857364: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (24afa8715391): /proc/driver/nvidia/version does not exist
2021-10-06 18:51:07.859120: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operation

# Train the model

In [14]:
batch_size = 128
epochs = 15

model.compile(loss = "categorical_crossentropy",optimizer="adam", metrics=["accuracy"])

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)


2021-10-06 18:51:08.281022: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2021-10-06 18:51:08.293637: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 2199995000 Hz


Epoch 1/15
422/422 [==============================] - 22s 51ms/step - loss: 0.7573 - accuracy: 0.7608 - val_loss: 0.0858 - val_accuracy: 0.9773
Epoch 2/15
422/422 [==============================] - 21s 49ms/step - loss: 0.1221 - accuracy: 0.9628 - val_loss: 0.0590 - val_accuracy: 0.9828
Epoch 3/15
422/422 [==============================] - 21s 49ms/step - loss: 0.0874 - accuracy: 0.9731 - val_loss: 0.0487 - val_accuracy: 0.9877
Epoch 4/15
422/422 [==============================] - 20s 48ms/step - loss: 0.0728 - accuracy: 0.9788 - val_loss: 0.0466 - val_accuracy: 0.9855
Epoch 5/15
422/422 [==============================] - 21s 49ms/step - loss: 0.0642 - accuracy: 0.9801 - val_loss: 0.0367 - val_accuracy: 0.9892
Epoch 6/15
422/422 [==============================] - 21s 49ms/step - loss: 0.0571 - accuracy: 0.9822 - val_loss: 0.0375 - val_accuracy: 0.9903
Epoch 7/15
422/422 [==============================] - 20s 48ms/step - loss: 0.0555 - accuracy: 0.9830 - val_loss: 0.0355 - val_accuracy:

# Evaluate the trained model

In [15]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.023548433557152748
Test accuracy: 0.9921000003814697
